[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/spin_transformations/spin_transformations.ipynb)

# Spin Transformations

This example computes spin transformations: bendings of a surface that keep every angle. A turn and a scale at each vertex act on the edges around it, and [Crane, Pinkall and Schröder (2011)](https://www.cs.cmu.edu/~kmcrane/Projects/SpinTransformations/paper.pdf) choose them so that the surface's mean curvature changes by a prescribed amount. Repeated, the transformation becomes a conformal curvature flow that rounds Spot the cow into a sphere; asked for a constant change, its fields are the spinor spherical harmonics, and they spin the sphere into Dirac spheres.

In the algebra of three-dimensional space the turn and scale at each vertex is modelled as a scaled rotor, an even multivector, and the vertices and edges are points, antivectors, intrinsic like all geometry, which in three dimensions are bivectors. Every operator is a sparse extensor, a linear map between fields on the vertices, edges and faces of a mesh in which each element couples to only a few others by a multivector: the Dirac operator couples each face to its corners by bivectors, so it takes even fields to even fields, and the energy of a rotor field, its area-weighted squared residual, `~residual * face_areas * residual`, couples vertices by even multivectors. The new edges come from sandwiching each old edge with the rotors along it, `rotors << edges`.

In the quaternion notation of Crane, Pinkall and Schröder this reads as the eigenvalue problem $(D - \rho)\,\lambda = \gamma\lambda$ of least $|\gamma|$ for a quaternion field $\lambda$, and each new edge is $\tilde e = \bar\lambda\, e\, \lambda$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga.algebras import VGA3D as ga
from numga.sparse import spdiag
from examples import instantiate
from examples.util.animation import save_figure, show_animation
from examples.surfaces.spin_transformations import render
from examples.surfaces.spin_transformations.core import spot

# Meshes and their incidences as sparse maps between fields, in the algebra of space.
mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar = mesh_module.Mesh, mesh_module.as_scalar
mv = mesh_module.ctx.multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Vertices and edges are points, antivectors, intrinsic like all geometry; the turn and scale at each vertex is a
# scaled rotor, an even multivector.
Point = ga.gatype.antivector()                      # Antivector-
Even = ga.gatype.even()                             # Even+
# A volume, the join of three points, is an intrinsic scalar, and its unit gives an ordinary number that orientation.
Volume = Scalar.derive.flipped_orientation          # Scalar-
unit_volume = mv(Volume, [1])                       # [] Scalar-

np.set_printoptions(precision=4, suppress=True)


## Operators between fields

A field is an extensor with one element per vertex, edge or face, such as `vertices`, `Point[V]`. An operator between fields is a sparse extensor, a linear map that couples each element of one field to a few elements of another, each coupling an extensor. Here the couplings are multivectors, nullary extensors, which act by their product: times a field, each coupling multiplies its input element into its output element, and the products are summed; times another operator it composes the two; and `~` is its reverse, with inputs and outputs swapped and every coupling reversed. The mesh's incidences say which elements couple. These operators couple through scalars.

The geometry is read off the vertices: the edges are `boundary * vertices`, and from them come the face areas, the vertex areas, a third of each face at each of its corners, the cotangent weights, and the cotangent Laplacian, `~boundary * edge_weights * boundary`, all as diagonal or sparse maps between fields.

In [ ]:
# A unit sphere, its icosahedron split so many times.
n_subdivisions = 3


In [ ]:
mesh, vertices = Mesh.icosphere(n_subdivisions)
print(vertices.gatype, mesh.edges.shape, mesh.faces.shape)


In [ ]:
def cotangent_weights(mesh: Mesh, sides: Point) -> Scalar:
    """Each edge's cotangent weight, `Scalar+[edges]`, is half the cotangents of the angles facing it, from the edge
    facing each corner, `[3] Point-[faces]`; a corner's cotangent is one edge's slice read along the other, over the
    size of the plane they span."""
    after, before = sides[[1, 2, 0]], sides[[2, 0, 1]]                         # [3] Point-[faces], Point-[faces]
    cotangents = -(after.dual_inverse() & before) / after.commutator(before).norm()   # [3] Scalar+[faces]
    return (~mesh.incidence[2, 1] * (cotangents / 2)).sum(axis=0)

In [ ]:
# The boundary, taking each edge's tail from its head, and the means over each edge and each face.
boundary = mesh.d[0]                                                           # Scalar+[edges <- vertices]
edge_means = mesh.incidence[1, 0].mean(axis=0)                                 # Scalar+[edges <- vertices]
face_means = mesh.incidence[2, 0].mean(axis=0)                                 # Scalar+[faces <- vertices]

edges = boundary * vertices                                                    # Point-[edges]
# The edge facing each corner of each face is one of the edges, signed as the face runs along it.
sides = mesh.boundary[2, 1] * edges                                            # [3] Point-[faces]
# A face's area is half the size of the plane two of its edges span.
areas = sides[1].commutator(sides[0]).norm() / 2                               # Scalar+[faces]
# The diagonal maps of the face areas and of the vertex areas, a third of each face at each of its corners.
face_areas, vertex_areas = spdiag(areas), spdiag(~face_means * areas)          # Scalar+[faces <- faces], Scalar+[vertices <- vertices]
# Each edge weighted by half the cotangents of the two angles facing it, and the cotangent Laplacian they make.
edge_weights = spdiag(cotangent_weights(mesh, sides))                          # Scalar+[edges <- edges]
laplacian = ~boundary * edge_weights * boundary                                # Scalar+[vertices <- vertices]

In [ ]:
print(boundary.gatype, boundary.shape)
print(edges.gatype)

## A change in mean curvature

The transformation is asked for by a change in each face's mean curvature, `curvature_change`, a scalar field on the faces. Here it grows along y, each face's centroid paired with the direction y, `centroids & mv.y`: the surface is to bend more on one side and less on the other, red where it is to bend more.

In [ ]:
# How strongly the mean curvature changes along y.
dipole = 0.5

In [ ]:
centroids = face_means * vertices                                       # Point-[faces]
curvature_change = (centroids & mv.y) * dipole                          # Scalar+[faces]

save_figure(render.draw_surface(mesh, vertices, curvature_change), "spin_transformations_curvature");

## The Dirac operator

The Dirac operator sends a field of even multivectors on the vertices to a field on the faces. Crane, Pinkall and Schröder build it edge by edge: each face takes the value at each corner by the edge that corner faces, over twice the face's area, so the couplings of `dirac` are bivectors and it takes even fields to even fields. The same operator is the geometric derivative, `derivative`, with each face's unit normal multiplied in on the right: on each face a field on the vertices is linear in between, and the coupling to each corner is the gradient of that corner's hat function, the edge the corner faces turned a quarter turn within the face, `edge | normal`. The transformation's rotors solve `dirac * field == potential * field`, the potential putting the curvature change on each face in the unit of volume, `unit_volume`, which gives it the intrinsic orientation of the edges it is added to. So the residual's couplings are even, and the energy of a field is its area-weighted squared residual, `~residual * face_areas * residual`, its couplings even too.

In the quaternion matrix notation of Crane, Pinkall and Schröder this reads as $D_{ij} = -\tfrac{1}{2\mathcal{A}_i} e_j$, where $e_j$ denotes the edge vector facing corner $j$ as an imaginary quaternion, its sign in Hamilton's convention opposite to that of the edge's bivector, $A = D - R$, and the energy $A^{H} M_F A$.

In [ ]:
# Each face takes each corner by the edge it faces, over twice the face's area; with the face's normal, the same
# couplings are the gradients of the corners' hat functions.
dirac = (spdiag(sides / (2 * areas)) * mesh.incidence[2, 0]).sum(axis=0)        # Point-[faces <- vertices]
face_normals = (sides[0] & sides[1]) / (2 * areas)                              # Vector-[faces]
derivative = (spdiag((sides | face_normals) / (2 * areas)) * mesh.incidence[2, 0]).sum(axis=0)   # Vector+[faces <- vertices]
# The residual of `dirac * field == potential * field`, and its area-weighted square.
residual = (spdiag(sides / (2 * areas) - curvature_change * unit_volume / 3) * mesh.incidence[2, 0]).sum(axis=0)   # Even-[faces <- vertices]
energy = ~residual * face_areas * residual                                      # Even+[vertices <- vertices]

In [ ]:
print(derivative.gatype, dirac.gatype)
print(energy.gatype, energy.shape)

## The rotor field

The rotors are the field of least energy per unit of vertex area, the least eigenpair of `energy` against the vertex areas. An eigensolve has to know what a linear map acts on, so `energy * Even` and `vertex_areas * Even` turn its multivectors into maps on even multivectors. The field is fixed only up to a constant factor, which would turn and scale the whole surface, so it is divided by its area-weighted mean. The colours show how much each face's mean rotor scales it, red where it grows and blue where it shrinks.

In [ ]:
# The least eigenfield of the energy against the vertex areas, divided by its area-weighted mean.
_, (field,) = (energy * Even).eigh(vertex_areas * Even, 1)                     # Even+[vertices]
rotors = field / ((vertex_areas * field).sites.sum() / areas.sites.sum())      # Even+[vertices]
scaling = (face_means * rotors).scalar_norm_squared() - 1                      # Scalar+[faces]

save_figure(render.draw_surface(mesh, vertices, scaling), "spin_transformations_scaling");

## The new surface

Each edge is turned and scaled by the field along it, the reverse sandwich `rotor << edge` with the rotor running linearly from one end to the other, integrated by Simpson's rule: the turns at the two ends and twice the turn at the middle. The turns at the ends are averaged as maps, `rotors << Point`, which does not depend on the sign of each rotor. The new vertices are those whose edges match the turned ones best, weighted by the cotangent Laplacian, which leaves the surface centred. The checkerboard is the sphere's: a conformal map keeps its squares square.

In the quaternion notation of Crane, Pinkall and Schröder this reads as $\tilde e_{ij} = \tfrac13 \bar\lambda_i e_{ij} \lambda_i + \tfrac16 \bar\lambda_i e_{ij} \lambda_j + \tfrac16 \bar\lambda_j e_{ij} \lambda_i + \tfrac13 \bar\lambda_j e_{ij} \lambda_j$.

In [ ]:
# Simpson's rule along each edge: the turns at its two ends, averaged as maps, and twice the turn at its middle.
turned_edges = ((edge_means * (rotors << Point))(edges) + 2 * ((edge_means * rotors) << edges)) / 3   # Point-[edges]
transformed_vertices = (laplacian * Point).lstsq(~boundary * edge_weights * turned_edges)   # Point-[vertices]

save_figure(render.draw_textured(mesh, transformed_vertices, vertices), "spin_transformations_deformed");

## Conformal curvature flow

Repeating the transformation, each time asking for a share of every face's departure from the mean curvature to be taken away, rounds a surface step by step while every step keeps its angles. Spot the cow, by Keenan Crane, draws in her legs, horns and ears and rounds into a sphere; a checkerboard laid on the cow keeps its squares square all the way, since no step changes an angle.

In [ ]:
# The share of each face's curvature departure each step removes, the steps, and the duration of a frame.
rate = 0.05
n_flow_steps = 60
duration_ms = 40

In [ ]:
def recentred(vertices: Point) -> Point:
    """The vertices are moved to their centre and scaled to unit mean distance from it, which is what a spin
    transformation leaves free."""
    vertices = vertices - vertices.sites.mean()
    return vertices / vertices.norm().sites.mean()


def rounding(mesh: Mesh, vertices: Point, n_steps: int, rate: float) -> Iterator[Point]:
    """The rounding yields the vertices at the start and after each step, each step one spin transformation that
    takes away a share `rate` of every face's departure from the area-weighted mean curvature."""
    edge_means = mesh.incidence[1, 0].mean(axis=0)                             # Scalar+[edges <- vertices]
    face_means = mesh.incidence[2, 0].mean(axis=0)                             # Scalar+[faces <- vertices]
    vertices = recentred(vertices)
    yield vertices
    for _ in range(n_steps):
        # The operators of the surface as it stands.
        edges = mesh.d[0] * vertices                                           # Point-[edges]
        sides = mesh.boundary[2, 1] * edges                                    # [3] Point-[faces]
        areas = sides[1].commutator(sides[0]).norm() / 2                       # Scalar+[faces]
        face_areas, vertex_areas = spdiag(areas), spdiag(~face_means * areas)  # Scalar+[faces <- faces], Scalar+[vertices <- vertices]
        edge_weights = spdiag(cotangent_weights(mesh, sides))                  # Scalar+[edges <- edges]
        laplacian = ~mesh.d[0] * edge_weights * mesh.d[0]                      # Scalar+[vertices <- vertices]
        # Each face's signed mean curvature: the cotangent Laplacian of the positions is the area-integrated
        # mean-curvature normal at each vertex, met with the vertex's normal, read as an ordinary
        # scalar in the unit of volume, over twice the vertex's area, and averaged onto the faces.
        normals = (~face_means * (sides[0] & sides[1])).normalized()            # Vector-[vertices]
        curvature = face_means * (((laplacian * vertices) & normals) * unit_volume / (2 * (~face_means * areas)))   # Scalar+[faces]
        change = -rate * (curvature - (curvature * areas).sites.sum() / areas.sites.sum())   # Scalar+[faces]
        # The spin transformation of that change, as above.
        residual = (spdiag(sides / (2 * areas) - change * unit_volume / 3) * mesh.incidence[2, 0]).sum(axis=0)   # Even-[faces <- vertices]
        _, (field,) = ((~residual * face_areas * residual) * Even).eigh(vertex_areas * Even, 1)   # Even+[vertices]
        rotors = field / ((vertex_areas * field).sites.sum() / areas.sites.sum())   # Even+[vertices]
        turned_edges = ((edge_means * (rotors << Point))(edges) + 2 * ((edge_means * rotors) << edges)) / 3   # Point-[edges]
        vertices = recentred((laplacian * Point).lstsq(~mesh.d[0] * edge_weights * turned_edges))
        yield vertices


cow, cow_vertices = spot()
show_animation(render.animate_rounding(cow, rounding(cow, cow_vertices, n_flow_steps, rate)), "spin_rounding", duration_ms)

## Dirac spheres

With `rho` a constant `n`, the fields of zero energy are the eigenfields of the Dirac operator itself. Their eigenvalues are the integers except −1, each with multiplicity one more than itself, and spinning the sphere by one gives a Dirac sphere. `dirac_spheres` runs the energy above with that constant and spins the sphere by each of the fields of least energy, one row per eigenvalue. The higher states are finer, so the sphere is split once more.

Crane, Pinkall and Schröder name the physics these fields come from (§6.2):

> Eigenfunctions of D on the unit sphere are called the spinor spherical harmonics, which can be used to compute fundamental solutions to the Dirac equation for the configuration of a spin-1/2 particle such as an electron in a spherically symmetric potential.

In [ ]:
# The finer sphere's subdivisions, the eigenvalues shown, and the states shown of each.
n_dirac_subdivisions = 4
eigenvalues = (1, 2, 3, -2, -3)
n_states = 3

In [ ]:
def dirac_spheres(mesh: Mesh, vertices: Point, eigenvalues: np.ndarray, n_states: int) -> tuple[Scalar, Point]:
    """For each constant curvature change, one per eigenvalue, the `n_states` fields of least energy have their
    energies and spin the mesh into surfaces with vertices `[eigenvalues, states] Point-[vertices]`; the fields are
    taken at unit size per unit of vertex area, as they come."""
    edge_means = mesh.incidence[1, 0].mean(axis=0)                             # Scalar+[edges <- vertices]
    face_means = mesh.incidence[2, 0].mean(axis=0)                             # Scalar+[faces <- vertices]
    edges = mesh.d[0] * vertices                                               # Point-[edges]
    sides = mesh.boundary[2, 1] * edges                                        # [3] Point-[faces]
    areas = sides[1].commutator(sides[0]).norm() / 2                           # Scalar+[faces]
    face_areas, vertex_areas = spdiag(areas), spdiag(~face_means * areas)      # Scalar+[faces <- faces], Scalar+[vertices <- vertices]
    edge_weights = spdiag(cotangent_weights(mesh, sides))                      # Scalar+[edges <- edges]
    laplacian = ~mesh.d[0] * edge_weights * mesh.d[0]                          # Scalar+[vertices <- vertices]

    # Every eigenvalue's constant change at once, a batch of residuals.
    constant = as_scalar(eigenvalues[:, None]) * unit_volume / 3   # [eigenvalues, 1] Scalar-
    residual = (spdiag(sides / (2 * areas) - constant) * mesh.incidence[2, 0]).sum(axis=-1)   # [eigenvalues] Even-[faces <- vertices]
    energies, eigenfields = ((~residual * face_areas * residual) * Even).eigh(vertex_areas * Even, n_states)   # [eigenvalues, states] Even+[vertices]
    turned_edges = ((edge_means * (eigenfields << Point))(edges) + 2 * ((edge_means * eigenfields) << edges)) / 3   # [eigenvalues, states] Point-[edges]
    return energies, (laplacian * Point).lstsq(~mesh.d[0] * edge_weights * turned_edges)


fine_mesh, fine_vertices = Mesh.icosphere(n_dirac_subdivisions)
dirac_energies, dirac_vertices = dirac_spheres(fine_mesh, fine_vertices, np.array(eigenvalues, dtype=float), n_states)   # [eigenvalues, states]

save_figure(render.draw_dirac(fine_mesh, dict(zip(eigenvalues, dirac_vertices)), fine_vertices), "dirac_spheres");

In [ ]:
# checks
# Crane's Dirac operator is the geometric derivative with each face's unit normal on the right, where it
# anticommutes with the in-plane gradients; and the geometric derivative of position is twice the volume
# element on every face: the dimension of the surface, with no curl.
np.testing.assert_allclose((spdiag(face_normals) * derivative + dirac).cells.kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((derivative * vertices).kernel, np.broadcast_to([0, 0, 0, 2], (len(mesh.faces), 4)), atol=1e-12)
# Every corner keeps its angle, up to the mesh's resolution.
def corner_cosines(vertices: Point) -> Scalar:
    """Each corner's cosine is one edge's slice read along the other, over both lengths."""
    sides = mesh.boundary[2, 1] * (boundary * vertices)                        # [3] Point-[faces]
    after, before = sides[[1, 2, 0]], sides[[2, 0, 1]]                         # [3] Point-[faces], Point-[faces]
    return -(after.dual_inverse() & before) / (after.norm() * before.norm())


np.testing.assert_allclose((corner_cosines(transformed_vertices) - corner_cosines(vertices)).to_array(), 0.0, atol=0.05)
# The Dirac spheres' fields have zero energy, up to the mesh's resolution against the spectrum's unit spacing.
np.testing.assert_allclose(dirac_energies.to_array(), 0.0, atol=0.05)